# `em_tools` Interactive Workbench

This notebook groups all core workflows of `em_tools` into structured chapters:

1. **Workspace Setup & Configuration** — Configure paths, initialize SQLite, and inspect available readers.
2. **Meter Tree & Master-Data Database (SQLite ↔ `.xlsx` Roundtrip)** — Inspect `objects`, `metering_points` (self-describing tree with auto-computed depth), and `estimation_rules`; export to Excel/LibreOffice, preview diffs (`dry_run`), commit validated changes with automatic backups, and manage restores.
3. **Importing Energy Data into Standardized Parquet** — Subchapters for each input format (`edyna_xlsx`, `csv_standard`, manual readings, and batch folder ingestion).
4. **Querying & Inspecting Standardized Parquet Data** — Filtered Polars reads and ad-hoc DuckDB SQL queries.
5. **Transformations, Data Quality Tests & Quarterly Reports (`dbt-duckdb`)** — Run recursive tree & residual models, execute data quality tests (including energy conservation), inspect quarterly marts, and export reports.
6. **Optional Demo Dataset Generator** — Populate an isolated demo workspace (`../data_demo`) featuring a mixed-class 3-level tree (`public_lighting` residual + `building` + `school` + `mobility` estimation share + `water`) so you can run Chapters 1–5 end-to-end.

> **Tip:** Run **Chapter 1** first so `cfg`, `repo`, and `store` are initialized for all subsequent chapters.

# 1 - Workspace Setup & Configuration

All local files live under a single `DATA_DIR` (`raw/`, `parquet/`, `db/`, `db/backups/`, `editor/`).
- Point `DATA_DIR` to `REPO_ROOT / "data"` for real operations, or `REPO_ROOT / "data_demo"` to experiment safely with the sample generator in Chapter 6.
- `DEFAULT_TIMEZONE` specifies the local timezone used when raw input files contain naive local timestamps (all timestamps are normalized to **UTC interval start** in Parquet).

In [1]:
from datetime import datetime, timezone
import os
from pathlib import Path
import subprocess
import sys

import duckdb
import polars as pl

from em_tools import (
    DEFAULT_REGISTRY,
    EdynaXlsxReader,
    EmToolsConfig,
    MasterDataRepository,
    ParquetMeasurementStore,
    RoundtripValidationError,
    StandardCsvReader,
    export_to_editor,
    import_from_editor,
    ingest_directory,
    ingest_file,
)

# Resolve repository root whether Jupyter is launched from root or notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DBT_DIR = REPO_ROOT / "dbt"

# Switch to REPO_ROOT / "data_demo" if you want to try Chapter 6 first
DATA_DIR = REPO_ROOT / "data"
DEFAULT_TIMEZONE = "Europe/Rome"

cfg = EmToolsConfig(data_dir=DATA_DIR, default_timezone=DEFAULT_TIMEZONE)
cfg.ensure_directories()

repo = MasterDataRepository(cfg.db_path, backups_dir=cfg.backups_dir)
repo.init()

store = ParquetMeasurementStore(
    cfg.parquet_dir,
    compression=cfg.parquet_compression,
    default_timezone=cfg.default_timezone,
)

print(f"Workspace directory : {cfg.data_dir}")
print(f"SQLite database     : {cfg.db_path}")
print(f"Parquet store       : {cfg.parquet_dir}")
print(f"Default editor file : {cfg.default_editor_path}")
print("\nRegistered readers:")
for r in DEFAULT_REGISTRY.list_readers():
    print(f"  - {r.name}: {r.description}")

Workspace directory : c:\Users\thoma\Documents\GitHub\em_tools\data
SQLite database     : c:\Users\thoma\Documents\GitHub\em_tools\data\db\masterdata.sqlite
Parquet store       : c:\Users\thoma\Documents\GitHub\em_tools\data\parquet
Default editor file : c:\Users\thoma\Documents\GitHub\em_tools\data\editor\masterdata_roundtrip.xlsx

Registered readers:
  - csv_standard: Generic tidy CSV reader (smart meters, sub-meters, water, gas, manual).
  - edyna_xlsx: Edyna 15-minute electricity .xlsx export (date + 96 quarter-hour columns).


# 2 - Managing the Master-Data Database

The local SQLite database (`masterdata.sqlite`) is the **Single Source of Truth** with three clean tables:
- **`objects`** — Buildings, schools, street lighting zones, water supply assets, etc. organized by `group` and `sub_group`.
- **`metering_points`** — Self-describing meter tree of **unlimited depth**: leave `parent_metering_point_id` blank for main meters (level 0), or set it to the parent meter ID for sub-meters. Each meter's `object_id` owns its measured value (leaf) **or its automatic residual ("the rest")** when it has sub-meters. Parent and child must share `energy_carrier` & `unit`, while their objects can belong to any `group` / `sub_group`.
- **`estimation_rules`** — Optional time-bounded (`valid_from` / `valid_to`) percentage splits allocating a share of a meter's net rest to unmetered objects.

## Master-Data Field Reference

Fill in the three sheets of `masterdata_roundtrip.xlsx` in the order **`objects` → `metering_points` → `estimation_rules`** (gray columns starting with `_` are auto-managed on export — do not edit them).

#### Sheet 1: `objects` (Buildings, Facilities & Consumer Groups)

| Field Name | Required | Type / Allowed Values | Description | Example |
| :--- | :--- | :--- | :--- | :--- |
| `object_id` | **Yes** (PK) | `text` (unique) | Stable unique identifier of the building, facility, or consumer group. | `OBJ-TOWNHALL` |
| `name` | **Yes** | `text` | Human-readable display name shown in quarterly reports. | `Town Hall` |
| `group` | **Yes** | `text` | Top-level category / reporting group (e.g., `Buildings`, `Education`, `Public Lighting`, `Water Supply`). *(Quote as `"group"` in ad-hoc SQL.)* | `Buildings` |
| `sub_group` | **Yes** | `text` | Second-level reporting subgroup within `group`. | `Administration` |
| `active` | No (default `1`) | `1` (active) or `0` (inactive) | Active flag. Removing a row from the sheet automatically sets `active = 0` on import (soft-delete). | `1` |
| `notes` | No | `text` | Optional free-text remarks (address, building notes, history). | `Main municipal office` |
| `_row_hash` | Auto (read-only) | `text` (managed) | System hash for fast unchanged-row detection — **do not edit**. | *(auto)* |


#### Sheet 2: `metering_points` (Self-Describing Meter Tree — Unlimited Depth)

| Field Name | Required | Type / Allowed Values | Description | Example |
| :--- | :--- | :--- | :--- | :--- |
| `metering_point_id` | **Yes** (PK) | `text` (unique) | Stable immutable meter ID matching `metering_point_id` in Parquet measurements (assign a new ID if physically replaced/rewired). | `MP-TOWN-MAIN` |
| `parent_metering_point_id` | No | `text` (FK → `metering_points`) or **blank** | **Leave empty for main meters (level 0).** Set to the parent meter's ID for sub-meters of any depth (level 1, 2, …). Must match parent's `energy_carrier` & `unit`; cycles are rejected. | `MP-TOWN-MAIN` *(or blank)* |
| `object_id` | **Yes** | `text` (FK → `objects`) | Consumer object owning this meter's net consumption: for leaf meters = 100% measured value; for parent meters = **automatic residual ("the rest")** after deducting direct sub-meters and estimation shares. | `OBJ-STREETLIGHTS` |
| `energy_carrier` | **Yes** | `electricity`, `water`, `gas`, `heating`, `other` | Physical medium measured by the meter (also used for Parquet partitioning). Must match parent meter. | `electricity` |
| `name` | **Yes** | `text` | Human-readable label for the meter. | `Main Town Meter` |
| `external_id` | No | `text` | Optional utility/provider identifier (e.g., POD code, meter serial number, OBIS reference). | `IT001E12345678` |
| `unit` | **Yes** | `text` | Standardized measurement unit. Must match parent meter. | `kWh` or `m3` |
| `meter_type` | **Yes** | `smartmeter`, `submeter`, `manual`, `virtual`, `pv_production` | Metering technology / role. Use `pv_production` for PV generators (`energy_carrier` must be `electricity`; set `parent_metering_point_id` to the grid meter for self-consumption or leave blank for standalone full feed-in). | `smartmeter` |
| `active` | No (default `1`) | `1` (active) or `0` (inactive) | Active flag (`0` = soft-deleted / decommissioned). | `1` |
| `valid_from` | No | `YYYY-MM-DD` | Optional installation or activation start date (ISO format). | `2026-01-01` |
| `valid_to` | No | `YYYY-MM-DD` | Optional decommissioning end date (`>= valid_from`; leave blank if active). | *(blank)* |
| `notes` | No | `text` | Optional remarks (cabinet location, CT ratio, residual explanation). | `Rest feeds street lighting` |
| `_meter_level` | Auto (read-only) | `int` (managed) | Auto-computed tree depth (`0` = main meter, `1` = sub-meter, `2` = sub-sub-meter…) — **do not edit**. | `1` |
| `_tree_path` | Auto (read-only) | `text` (managed) | Auto-computed path from root meter to this meter — **do not edit**. | `MP-TOWN-MAIN > MP-BLDG` |
| `_row_hash` | Auto (read-only) | `text` (managed) | System hash for fast unchanged-row detection — **do not edit**. | *(auto)* |


#### Sheet 3: `estimation_rules` (Optional Time-Bounded Percentage Splits)

| Field Name | Required | Type / Allowed Values | Description | Example |
| :--- | :--- | :--- | :--- | :--- |
| `rule_id` | Auto on new rows (PK) | positive `int` or **blank** | **Leave empty when adding a new rule** — auto-assigned on import. Keep existing integer IDs unchanged when editing. | *(blank for new)* / `1` |
| `source_metering_point_id` | **Yes** | `text` (FK → `metering_points`) | Meter whose net consumption (after subtracting direct sub-meters) is split by this rule. | `MP-TOWN-MAIN` |
| `target_object_id` | **Yes** | `text` (FK → `objects`) | Unmetered consumer object receiving the estimated share (may belong to any `group` / `sub_group`). | `OBJ-EV` |
| `factor` | **Yes** | `float` in `(0, 1]` | Fraction of the source meter's net rest allocated to `target_object_id` (e.g., `0.20` = 20%). Active overlapping rules per source meter must sum `<= 1.0`; any remainder `(1 - sum)` stays automatically on the source meter's `object_id`. | `0.20` |
| `valid_from` | **Yes** | `YYYY-MM-DD` | Inclusive start date when this estimation factor takes effect (ISO format). | `2026-01-01` |
| `valid_to` | No | `YYYY-MM-DD` | Inclusive end date (`>= valid_from`). Leave blank for open-ended rules. | `2026-12-31` *(or blank)* |
| `active` | No (default `1`) | `1` (active) or `0` (inactive) | Active flag. Removing a row soft-deletes it (`active = 0`) on import. | `1` |
| `notes` | No | `text` | Optional justification or calculation basis for the estimated factor. | `20% of residual for EV charger` |
| `_row_hash` | Auto (read-only) | `text` (managed) | System hash for fast unchanged-row detection — **do not edit**. | *(auto)* |



## Inspect Current SQLite Tables

In [9]:
for table_name in ("objects", "metering_points", "estimation_rules"):
    rows = repo.fetch_table(table_name)
    print(f"=== {table_name} ({len(rows)} rows) ===")
    if rows:
        display(pl.DataFrame(rows))
    else:
        print("(empty)\n")

=== objects (4 rows) ===


object_id,name,group,sub_group,active,notes
str,str,str,str,i64,null
"""GEB-001""","""Rathaus""","""Gebäude""","""Verwaltung""",1,null
"""GEB-002""","""KG Laurin""","""Gebäude""","""Kindergärten""",1,null
"""GEB-003""","""Mittelschule""","""Gebäude""","""Schulen""",1,null
"""GEB-004""","""Knabenschule""","""Gebäude""","""Schulen""",1,null


=== metering_points (2 rows) ===


metering_point_id,parent_metering_point_id,object_id,energy_carrier,name,external_id,unit,meter_type,active,valid_from,valid_to,notes
str,str,str,str,str,null,str,str,i64,str,null,null
"""MET-001""",null,"""GEB-001""","""electricity""","""Anschluss EDYNA""",null,"""kWh""","""smartmeter""",1,"""2026-01-01""",null,null
"""MET-002""","""MET-001""","""GEB-001""","""electricity""","""Unterzähler TW Pumpe""",null,"""kWh""","""submeter""",1,"""2026-01-01""",null,null


=== estimation_rules (1 rows) ===


rule_id,source_metering_point_id,target_object_id,factor,valid_from,valid_to,active,notes
i64,str,str,f64,str,null,i64,null
1,"""MET-002""","""GEB-004""",0.5,"""2026-01-01""",null,1,null


### 2.2 Export Master Data to Excel / LibreOffice (`.xlsx`)

Generates a formatted `.xlsx` workbook containing `README_EDITOR`, `objects`, `metering_points` (including auto-computed read-only `_meter_level` and `_tree_path` helper columns), and `estimation_rules` with dropdown validations and `_row_hash`.

**Editing rules:**
- Leave `parent_metering_point_id` **empty** for main meters (level 0); set it to the parent meter ID for sub-meters of any depth.
- Leave `rule_id` **empty** on newly added rows in `estimation_rules` — primary keys are assigned automatically on import.
- Do **not** edit gray managed columns (`_meter_level`, `_tree_path`, `_row_hash`).
- Removing a row marks it inactive (`active = 0`, soft-delete) on import to preserve history.

In [16]:
EDITOR_XLSX = cfg.default_editor_path

exported_path = export_to_editor(cfg.db_path, EDITOR_XLSX)
print(f"Exported master data to: {exported_path}")
print("Open this file in Excel or LibreOffice Calc, make your edits, save, and continue to 2.3.")

Exported master data to: c:\Users\thoma\Documents\GitHub\em_tools\data\editor\masterdata_roundtrip.xlsx
Open this file in Excel or LibreOffice Calc, make your edits, save, and continue to 2.3.


### 2.3 Preview & Validate Changes (`dry_run=True`)

Validates all edits without touching SQLite: row-hash fast skip, column-by-column diff, foreign keys, matching `energy_carrier` & `unit` between parent and child meters, DFS cycle detection on `metering_points`, overlapping estimation factor sums `<= 1.0`, and soft-delete preview.

In [12]:
try:
    dry_report = import_from_editor(
        cfg.db_path,
        EDITOR_XLSX,
        dry_run=True,
        backups_dir=cfg.backups_dir,
    )
    print(dry_report.format_summary())
except RoundtripValidationError as exc:
    print("VALIDATION FAILED — fix the following issues in the workbook:")
    print(exc)

Roundtrip import [DRY-RUN]:
  - objects: +0 insert(s), ~0 update(s), -0 soft-delete(s), =4 unchanged
  - metering_points: +1 insert(s), ~0 update(s), -0 soft-delete(s), =2 unchanged
  - estimation_rules: +1 insert(s), ~0 update(s), -0 soft-delete(s), =1 unchanged


### 2.4 Commit Validated `.xlsx` Changes into SQLite

Creates a timestamped backup in `data/db/backups/`, commits all inserts (parent-before-child + auto-assigned `rule_id`s), updates, and soft-deletes atomically, and optionally re-exports the workbook with refreshed `_meter_level`, `_tree_path`, `rule_id`, and `_row_hash` values.

In [13]:
RE_EXPORT_AFTER_COMMIT = True

try:
    commit_report = import_from_editor(
        cfg.db_path,
        EDITOR_XLSX,
        dry_run=False,
        backups_dir=cfg.backups_dir,
    )
    print(commit_report.format_summary())
    if RE_EXPORT_AFTER_COMMIT and commit_report.has_changes:
        export_to_editor(cfg.db_path, EDITOR_XLSX)
        print(f"Re-exported refreshed workbook (with tree levels, IDs & hashes) to: {EDITOR_XLSX}")
except RoundtripValidationError as exc:
    print("VALIDATION FAILED — SQLite database was NOT modified:")
    print(exc)

Roundtrip import [COMMITTED]:
  - objects: +0 insert(s), ~0 update(s), -0 soft-delete(s), =4 unchanged
  - metering_points: +1 insert(s), ~0 update(s), -0 soft-delete(s), =2 unchanged
  - estimation_rules: +1 insert(s), ~0 update(s), -0 soft-delete(s), =1 unchanged
  Assigned new rule_id(s): [2]
  Pre-commit DB backup: c:\Users\thoma\Documents\GitHub\em_tools\data\db\backups\masterdata_20261125T175658Z_pre_import.sqlite
Re-exported refreshed workbook (with tree levels, IDs & hashes) to: c:\Users\thoma\Documents\GitHub\em_tools\data\editor\masterdata_roundtrip.xlsx


### 2.5 List & Restore SQLite Backups

In [ ]:
backups = sorted(cfg.backups_dir.glob("*.sqlite"))
print(f"Found {len(backups)} backup(s) in {cfg.backups_dir}:")
for b in backups:
    print(f"  - {b.name}")

# To restore a specific backup, uncomment and set the filename:
# repo.restore_backup(cfg.backups_dir / "masterdata_YYYYMMDDTHHMMSSZ_pre_import.sqlite")

---
## Chapter 3: Importing Energy Consumption Data into Standardized Parquet

All readers normalize raw files into the canonical schema (UTC interval start, typed base + nullable carrier-specific columns) and upsert idempotently into:
`data/parquet/energy_carrier=<energy_carrier>/year=<YYYY>/measurements.parquet`

Reporting groups (`group`, `sub_group`) live strictly on `objects` in SQLite, so reorganizing assets never touches any Parquet file.

### 3.0 Auto-Ingest Raw Folder (Automatic Filename & Format Detection)

Scans `cfg.raw_dir`, automatically picks the matching reader + channel from each **filename + extension**, resolves `external_id` (e.g., Edyna POD) against SQLite `metering_points`, and upserts all matched files into Parquet (ignoring Excel `~$` lock files and reporting unrecognized files as skipped).

In [ ]:
batch_result = ingest_directory(cfg.raw_dir, config=cfg, recursive=False)
print(batch_result.format_summary())

### 3.1 Import Single Edyna Electricity `.xlsx` Export (`edyna_xlsx`)

Automatically recognizes all three Edyna filename conventions (local `Europe/Rome` with exact EU summer/winter DST handling — `92` / `96` / `100` quarter-hour UTC intervals/day):
- `ATTIVA_<POD>_<YYYYMM>.xlsx` → `grid_import` (`external_id = <POD>`)
- `ATTIVA_S_<POD>_<YYYYMM>.xlsx` → `grid_export` (`external_id = <POD>`)
- `ATTIVA_<POD>P_<YYYYMM>.xlsx` → `pv_production` (`external_id = <POD>P` or linked PV meter under `<POD>`)

In [ ]:
EDYNA_FILE = cfg.raw_dir / "ATTIVA_IT005E00143758_202601.xlsx"  # <-- adjust to your raw Edyna file

if EDYNA_FILE.exists():
    res = ingest_file(EDYNA_FILE, config=cfg)
    print(
        f"Ingested {res.rows_input} rows from {res.file_path.name} "
        f"(reader={res.reader_name}, mp={list(res.metering_point_ids)}, channel={list(res.channels)})"
    )
    for p in res.partition_files:
        print(f"  -> {p}")
else:
    print(f"File not found: {EDYNA_FILE} (place an ATTIVA_*.xlsx file in {cfg.raw_dir})")

### 3.2 Import Generic / Sub-Meter / Water / Gas CSV (`csv_standard`)

Use `StandardCsvReader` for tidy CSV exports across any `energy_carrier` (`electricity`, `water`, `gas`, `heating`, `other`). Optional fields (`peak_power_kw`, `flow_rate`, `pressure_bar`, `burn_duration_min`) are preserved automatically when present.

In [ ]:
CSV_FILE = cfg.raw_dir / "water_sample.csv"  # <-- adjust to your CSV file

if CSV_FILE.exists():
    csv_reader = StandardCsvReader()
    df_csv = csv_reader.read(
        CSV_FILE,
        separator=";",
        decimal_comma=True,
        default_timezone=cfg.default_timezone,
        metering_point_id="MP-WATER-1",  # optional if column already exists in CSV
        energy_carrier="water",          # optional if column already exists in CSV
        unit="m3",                       # optional if column already exists in CSV
        interval_minutes=15,
    )
    display(df_csv.head(5))
    summary = store.write(df_csv)
    print(f"Upserted {summary.rows_input} rows into {summary.partitions_written} partition(s).")
else:
    print(f"File not found: {CSV_FILE} (place a CSV there or run Chapter 6 demo)")

### 3.3 Import Manual Meter Readings (Cumulative Counter Values)

Manual counter readings are stored with `reading_type="cumulative"`, `quality="manual"`, and `interval_minutes=None`, keeping counter totals isolated from 15-minute `interval` consumptions.

In [ ]:
MANUAL_CSV = cfg.raw_dir / "manual_readings.csv"  # <-- adjust to your manual readings CSV

if MANUAL_CSV.exists():
    df_manual = StandardCsvReader().read(
        MANUAL_CSV,
        separator=",",
        decimal_comma=False,
        default_timezone=cfg.default_timezone,
        interval_minutes=None,
        reading_type="cumulative",
        quality="manual",
    )
    display(df_manual.head(5))
    summary = store.write(df_manual)
    print(f"Upserted {summary.rows_input} manual reading(s).")
else:
    print(f"File not found: {MANUAL_CSV}")
    print("Expected CSV columns: timestamp,metering_point_id,energy_carrier,value,unit")

### 3.4 Batch-Import a Folder of Files

In [ ]:
BATCH_DIR = cfg.raw_dir / "edyna_batch"  # <-- folder containing multiple files of one format
BATCH_GLOB = "*.xlsx"
BATCH_READER_NAME = "edyna_xlsx"

if BATCH_DIR.exists():
    reader = DEFAULT_REGISTRY.get(BATCH_READER_NAME)
    files = sorted(BATCH_DIR.glob(BATCH_GLOB))
    print(f"Found {len(files)} file(s) in {BATCH_DIR}")
    for fpath in files:
        df_file = reader.read(fpath, default_timezone=cfg.default_timezone)
        res = store.write(df_file)
        print(f"  {fpath.name}: +{res.rows_input} rows -> {res.partitions_written} partition(s)")
else:
    print(f"Batch directory not found: {BATCH_DIR} (create it to batch-ingest multiple files)")

---
## Chapter 4: Querying & Inspecting Standardized Parquet Data

### 4.1 Filtered Polars Reads via `ParquetMeasurementStore`

In [ ]:
df_all = store.read(
    # energy_carrier="electricity",
    # year=2026,
    # metering_point_ids=["MP-TOWN-MAIN"],
    # start_utc=datetime(2026, 1, 1, tzinfo=timezone.utc),
    # end_utc=datetime(2026, 4, 1, tzinfo=timezone.utc),
)

print(f"Total rows matched: {df_all.height}")
if not df_all.is_empty():
    overview = (
        df_all.group_by(["energy_carrier", "metering_point_id", "channel", "reading_type", "unit"])
        .agg(
            pl.len().alias("row_count"),
            pl.col("timestamp").min().alias("first_utc"),
            pl.col("timestamp").max().alias("last_utc"),
            pl.col("value").sum().round(3).alias("value_sum"),
        )
        .sort(["energy_carrier", "metering_point_id", "channel"])
    )
    display(overview)

### 4.2 Ad-Hoc SQL Queries Directly on Parquet with DuckDB

In [ ]:
parquet_files = list(cfg.parquet_dir.glob("energy_carrier=*/year=*/*.parquet"))
if parquet_files:
    parquet_glob = (cfg.parquet_dir / "energy_carrier=*" / "year=*" / "*.parquet").as_posix()
    with duckdb.connect() as con:
        df_sql = con.execute(
            f"""
            select
                energy_carrier,
                metering_point_id,
                channel,
                count(*) as intervals,
                round(sum(value), 2) as total_value,
                max(peak_power_kw) as max_peak_power_kw
            from read_parquet('{parquet_glob}', hive_partitioning=false)
            group by 1, 2, 3
            order by 1, 2, 3
            """
        ).pl()
    display(df_sql)
else:
    print(f"No Parquet files found yet under {cfg.parquet_dir}")

---
## Chapter 5: Transformations, Data Quality Tests & Quarterly Reports (`dbt-duckdb`)

`dbt-duckdb` attaches `masterdata.sqlite` (read-only), recursively derives meter levels (`int_metering_point_tree`), deducts direct children across arbitrary depth, applies `estimation_rules`, automatically assigns the remaining rest to each parent's `object_id`, and builds the quarterly reporting marts.

### 5.1 Run dbt Models (`staging` → `intermediate` → `marts`)

In [ ]:
DUCKDB_PATH = cfg.data_dir / "em_tools.duckdb"
DUCKDB_EXT_DIR = REPO_ROOT / ".duckdb_extensions"

def build_dbt_env() -> dict[str, str]:
    env = os.environ.copy()
    env["EM_TOOLS_DUCKDB_PATH"] = DUCKDB_PATH.as_posix()
    env["EM_TOOLS_SQLITE_PATH"] = cfg.db_path.as_posix()
    env["EM_TOOLS_PARQUET_GLOB"] = (
        cfg.parquet_dir / "energy_carrier=*" / "year=*" / "*.parquet"
    ).as_posix()
    env["EM_TOOLS_DUCKDB_EXT_DIR"] = DUCKDB_EXT_DIR.as_posix()
    return env

def run_dbt(subcommand: str) -> int:
    cmd = [
        sys.executable,
        "-m",
        "dbt.cli.main",
        subcommand,
        "--project-dir",
        str(DBT_DIR),
        "--profiles-dir",
        str(DBT_DIR),
    ]
    proc = subprocess.run(cmd, env=build_dbt_env(), capture_output=True, text=True, check=False)
    print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    return proc.returncode

rc = run_dbt("run")
print(f"dbt run exited with code {rc}")

### 5.2 Run dbt Data Quality & Energy Conservation Tests (`dbt test`)

Executes all schema integrity tests plus 7 custom SQL checks:
- Duplicate interval keys
- Negative interval consumptions
- Negative parent residuals (sub-meters exceeding parent busbar supply)
- Estimation factor sums exceeding `1.0`
- Missing 15-minute intervals on active days (DST-aware: `92`, `96`, or `100` intervals/day)
- **PV export plausibility** (`grid_export <= pv_production` in every 15-minute interval)
- **Energy conservation per root meter** (`sum(subtree allocations) == root_grid_import - root_grid_export + subtree_pv_production`)

In [ ]:
rc = run_dbt("test")
print(f"dbt test exited with code {rc}")

### 5.3 Inspect Quarterly Reporting Marts & Export Report

In [ ]:
if DUCKDB_PATH.exists():
    with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
        df_quarterly = con.execute("select * from mart_quarterly_consumption").pl()
        df_pv = con.execute("select * from mart_quarterly_pv_summary").pl()
        df_plausibility = con.execute("select * from mart_plausibility_summary").pl()

    print("=== Quarterly Consumption Mart (by Energy Carrier, Group & Sub-Group) ===")
    display(df_quarterly)

    print("=== Quarterly PV Summary Mart (Production, Feed-In, Self-Consumption & Autarky) ===")
    display(df_pv)

    print("=== Quarterly Plausibility & Meter Tree Summary ===")
    display(df_plausibility)

    # Optional: export quarterly report to CSV
    # report_out = cfg.data_dir / "quarterly_consumption_report.csv"
    # df_quarterly.write_csv(report_out)
    # print(f"Saved report to {report_out}")
else:
    print(f"DuckDB file not found yet: {DUCKDB_PATH}. Run cell 5.1 first.")

---
## Chapter 6: Optional Demo Dataset Generator

Generates a complete demo workspace under `../data_demo` showcasing a **mixed-group 3-level meter tree + PV plants**:
- `MP-TOWN-MAIN` (`grid_import=750 kWh`, `grid_export=150 kWh`, L0, rest → `OBJ-STREETLIGHTS` [`Public Lighting / Center`])
  - `MP-TOWN-PV` (`pv_production=400 kWh`, L1, behind-the-meter PV → `self_consumption=250 kWh`, total busbar supply = `1000 kWh`)
  - `MP-BLDG` (`500 kWh`, L1, rest → `OBJ-TOWNHALL` [`Buildings / Administration`])
    - `MP-LIB` (`120 kWh`, L2, measured → `OBJ-LIBRARY` [`Buildings / Culture`])
  - `MP-SCHOOL` (`300 kWh`, L1, measured → `OBJ-SCHOOL` [`Education / Primary Schools`])
- `estimation_rules`: `20%` of `MP-TOWN-MAIN`'s net rest (`200 kWh`) → `OBJ-EV` [`Mobility / Charging`] (`40 kWh`), leaving `160 kWh` automatically for `OBJ-STREETLIGHTS`
- Standalone full feed-in PV plant `MP-PV-SOLO` (`600 kWh`, `pv_production` & `grid_export` → `OBJ-SOLAR-PARK`)
- Plus `MP-WATER-1` (`48 m3`, `water` → `OBJ-RESERVOIR` [`Water Supply / Reservoirs`])

After running this cell, set `DATA_DIR = REPO_ROOT / "data_demo"` in **Chapter 1** and run Chapters 1–5 end-to-end!

In [15]:
from datetime import timedelta
import openpyxl

DEMO_DIR = REPO_ROOT / "data_demo"
demo_cfg = EmToolsConfig(data_dir=DEMO_DIR, default_timezone=DEFAULT_TIMEZONE)
demo_cfg.ensure_directories()

demo_repo = MasterDataRepository(demo_cfg.db_path, backups_dir=demo_cfg.backups_dir)
demo_repo.init()

# 1. Populate master data via the real .xlsx roundtrip workflow
export_to_editor(demo_cfg.db_path, demo_cfg.default_editor_path)
wb = openpyxl.load_workbook(demo_cfg.default_editor_path)

ws_obj = wb["objects"]
if ws_obj.max_row == 1:
    ws_obj.append(["OBJ-STREETLIGHTS", "Center Streetlights", "Public Lighting", "Center", 1, None])
    ws_obj.append(["OBJ-TOWNHALL", "Town Hall", "Buildings", "Administration", 1, None])
    ws_obj.append(["OBJ-LIBRARY", "Public Library", "Buildings", "Culture", 1, None])
    ws_obj.append(["OBJ-SCHOOL", "Primary School", "Education", "Primary Schools", 1, None])
    ws_obj.append(["OBJ-EV", "EV Charging Station", "Mobility", "Charging", 1, None])
    ws_obj.append(["OBJ-RESERVOIR", "North Reservoir", "Water Supply", "Reservoirs", 1, None])
    ws_obj.append(["OBJ-SOLAR-PARK", "Depot PV Roof", "Energy Generation", "PV Plants", 1, None])

    ws_mp = wb["metering_points"]
    ws_mp.append(["MP-TOWN-MAIN", None, "OBJ-STREETLIGHTS", "electricity", "Main Town Grid Meter", "POD-MAIN", "kWh", "smartmeter", 1, "2026-01-01", None, None])
    ws_mp.append(["MP-TOWN-PV", "MP-TOWN-MAIN", "OBJ-STREETLIGHTS", "electricity", "Town Hall PV Array", "M2-TOWN", "kWh", "pv_production", 1, "2026-01-01", None, None])
    ws_mp.append(["MP-BLDG", "MP-TOWN-MAIN", "OBJ-TOWNHALL", "electricity", "Town Hall Sub-meter", "SUB-HALL", "kWh", "submeter", 1, "2026-01-01", None, None])
    ws_mp.append(["MP-LIB", "MP-BLDG", "OBJ-LIBRARY", "electricity", "Library Sub-sub-meter", "SUB-LIB", "kWh", "submeter", 1, "2026-01-01", None, None])
    ws_mp.append(["MP-SCHOOL", "MP-TOWN-MAIN", "OBJ-SCHOOL", "electricity", "School Sub-meter", "SUB-SCH", "kWh", "submeter", 1, "2026-01-01", None, None])
    ws_mp.append(["MP-PV-SOLO", None, "OBJ-SOLAR-PARK", "electricity", "Depot Full Feed-In PV", "M2-DEPOT", "kWh", "pv_production", 1, "2026-01-01", None, None])
    ws_mp.append(["MP-WATER-1", None, "OBJ-RESERVOIR", "water", "Reservoir Water Meter", "W-1", "m3", "smartmeter", 1, "2026-01-01", None, None])

    ws_est = wb["estimation_rules"]
    ws_est.append([None, "MP-TOWN-MAIN", "OBJ-EV", 0.20, "2026-01-01", None, 1, "20% of net rest"])

    wb.save(demo_cfg.default_editor_path)
wb.close()

demo_report = import_from_editor(
    demo_cfg.db_path, demo_cfg.default_editor_path, backups_dir=demo_cfg.backups_dir
)
export_to_editor(demo_cfg.db_path, demo_cfg.default_editor_path)
print(demo_report.format_summary())

# 2. Write sample raw files (Edyna .xlsx + Water CSV)
edyna_wb = openpyxl.Workbook()
edyna_ws = edyna_wb.active
edyna_ws.title = "MP-TOWN-MAIN"
slots = [f"{h:02d}{m:02d}" for h in range(24) for m in (0, 15, 30, 45)]
edyna_ws.append(["GIORNO", *slots, "TOT", "F1", "F2", "F3"])
edyna_ws.append(["10/02/2026", *[f"{1000.0 / 96.0:.6f}".replace(".", ",") for _ in range(96)], 1000, 0, 0, 0])
edyna_wb.save(demo_cfg.raw_dir / "edyna_sample.xlsx")
edyna_wb.close()

day = datetime(2026, 2, 10, 0, 0)
csv_lines = ["timestamp;value;flow_rate;pressure_bar"]
for i in range(96):
    ts = (day + timedelta(minutes=15 * i)).strftime("%Y-%m-%d %H:%M:%S")
    csv_lines.append(f"{ts};0,5;2,0;4,1")
(demo_cfg.raw_dir / "water_sample.csv").write_text("\n".join(csv_lines) + "\n", encoding="utf-8")

# 3. Populate Parquet directly for all demo meters & PV channels
demo_store = ParquetMeasurementStore(demo_cfg.parquet_dir, default_timezone=DEFAULT_TIMEZONE)
demo_rows = []
for mp, carrier, channel, unit, total, peak in [
    ("MP-TOWN-MAIN", "electricity", "grid_import", "kWh", 750.0, 48.0),
    ("MP-TOWN-MAIN", "electricity", "grid_export", "kWh", 150.0, None),
    ("MP-TOWN-PV", "electricity", "pv_production", "kWh", 400.0, None),
    ("MP-BLDG", "electricity", "consumption", "kWh", 500.0, 28.0),
    ("MP-LIB", "electricity", "consumption", "kWh", 120.0, 9.0),
    ("MP-SCHOOL", "electricity", "consumption", "kWh", 300.0, 18.0),
    ("MP-PV-SOLO", "electricity", "pv_production", "kWh", 600.0, None),
    ("MP-PV-SOLO", "electricity", "grid_export", "kWh", 600.0, None),
    ("MP-WATER-1", "water", "consumption", "m3", 48.0, None),
]:
    for i in range(96):
        demo_rows.append(
            {
                "timestamp": day + timedelta(minutes=15 * i),
                "interval_minutes": 15,
                "metering_point_id": mp,
                "energy_carrier": carrier,
                "channel": channel,
                "value": total / 96.0,
                "unit": unit,
                "peak_power_kw": peak,
                "source": "demo_generator",
            }
        )

demo_summary = demo_store.write(pl.DataFrame(demo_rows))
print(f"\nDemo workspace ready at: {DEMO_DIR}")
print(f"Wrote {demo_summary.rows_stored} rows across {demo_summary.partitions_written} Parquet partition(s).")

Roundtrip import [COMMITTED]:
  - objects: +6 insert(s), ~0 update(s), -0 soft-delete(s), =0 unchanged
  - metering_points: +5 insert(s), ~0 update(s), -0 soft-delete(s), =0 unchanged
  - estimation_rules: +1 insert(s), ~0 update(s), -0 soft-delete(s), =0 unchanged
  Assigned new rule_id(s): [1]
  Pre-commit DB backup: c:\Users\thoma\Documents\GitHub\em_tools\data_demo\db\backups\masterdata_20261125T180422Z_pre_import.sqlite

Demo workspace ready at: c:\Users\thoma\Documents\GitHub\em_tools\data_demo
Wrote 480 rows across 2 Parquet partition(s).
